In [ ]:
!pip install factor_analyzer

import numpy as np
from sklearn.preprocessing import StandardScaler
from factor_analyzer.factor_analyzer import (
    calculate_kmo,
    calculate_bartlett_sphericity
)
from pathlib import Path
from sklearn.feature_selection import VarianceThreshold

In [31]:
X = np.load("/content/z_img.npy")

Paso 0

In [32]:
print("Cargado z_img:", X.shape)

Cargado z_img: (2450, 512)


In [33]:
X_uniq = np.unique(X, axis=0)
print(f"Únicos: {X_uniq.shape[0]} de {X.shape[0]}")

Únicos: 1095 de 2450


In [ ]:
stds_raw = X_uniq.std(axis=0)
vivas = stds_raw > 1e-6
X_clean = X_uniq[:, vivas]
print(f"Dimensiones vivas: {X_clean.shape[1]} de {X_uniq.shape[1]} (eliminadas {(~vivas).sum()})")
print(f"Proporción n/p: {X_clean.shape[0]/X_clean.shape[1]:.1f}:1")

In [35]:
X_afe = StandardScaler().fit_transform(X_clean)

In [36]:
assert (X_afe.std(axis=0) > 0).all(), "Aún hay columnas muertas"
corr = np.corrcoef(X_afe, rowvar=False)
assert not np.isnan(corr).any(), "Hay NaN en la matriz de correlación"
print("Matriz de correlación limpia (sin NaN)")

Matriz de correlación limpia (sin NaN)


In [ ]:
chi2, p = calculate_bartlett_sphericity(X_afe)
print(f"\nBartlett: chi²={chi2:.1f}  p={p:.3e}")
kmo_vars, kmo_global = calculate_kmo(X_afe)
print(f"KMO global: {kmo_global:.4f}")
print(f"KMO por variable -> media {kmo_vars.mean():.3f} | min {kmo_vars.min():.3f} | max {kmo_vars.max():.3f}")

In [38]:
print("Variables con KMO < 0.5:", (kmo_vars < 0.5).sum())
print("Variables con KMO < 0.6:", (kmo_vars < 0.6).sum())

Variables con KMO < 0.5: 1
Variables con KMO < 0.6: 1


## Pruebas de factorabilidad (KMO y Bartlett)

In [39]:
from factor_analyzer import FactorAnalyzer
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
fa_explore = FactorAnalyzer(rotation=None, method='principal')
fa_explore.fit(X_afe)

In [41]:
ev, _ = fa_explore.get_eigenvalues()
var_ratio = ev / ev.sum()
cumvar = np.cumsum(var_ratio)

In [ ]:
m_kaiser = int((ev > 1).sum())

m_80 = int(np.argmax(cumvar >= 0.80) + 1)
print(f"Kaiser (autovalor>1) : {m_kaiser} factores")
print(f"Para >=80% varianza  : {m_80} factores  (cumvar={cumvar[m_80-1]:.3f})")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].plot(range(1, 31), ev[:30], marker='o')
ax[0].axhline(1, ls='--', color='red', label='Kaiser')
ax[0].set(xlabel="Componente", ylabel="Autovalor", title="Scree Plot (primeros 30)")
ax[0].legend(); ax[0].grid(alpha=0.3)

ax[1].plot(range(1, len(cumvar)+1), cumvar, marker='.')
ax[1].axhline(0.80, ls='--', color='green', label='80%')
ax[1].axvline(m_80, ls=':', color='gray')
ax[1].set(xlabel="Nº componentes", ylabel="Varianza acumulada", title="Varianza acumulada")
ax[1].legend(); ax[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

## Selección del número de factores

In [ ]:
m = 16

fa = FactorAnalyzer(n_factors=m, method='principal', rotation='varimax')
fa.fit(X_afe)

var_total, prop_var, cum_prop = fa.get_factor_variance()
print(f"Varianza acumulada con {m} factores (rotados): {cum_prop[-1]:.3f}")

import pandas as pd
loadings = pd.DataFrame(fa.loadings_, columns=[f"F{i+1}" for i in range(m)])
loadings["comunalidad"] = fa.get_communalities()
print("\nComunalidades:")
print(loadings["comunalidad"].describe())

n_cargas = (loadings.filter(like="F").abs() > 0.4).sum(axis=1)
print("\nDimensiones según nº de factores en que cargan fuerte (|carga|>0.4):")
print(n_cargas.value_counts().sort_index())

In [45]:
import pandas as pd

tabla = pd.DataFrame({
    "Cargan fuerte en…": [
        "0 factores",
        "1 factor",
        "2 factores",
        "3 factores"
    ],
    "Nº dimensiones": [
        28,
        306,
        159,
        12
    ]
})

display(tabla)

,Cargan fuerte en…,Nº dimensiones
0,0 factores,28
1,1 factor,306
2,2 factores,159
3,3 factores,12


## Análisis de cargas cruzadas

In [ ]:
scores = fa.transform(X_afe)
print("Scores:", scores.shape)

print("Media:", scores.mean(), "| Std:", scores.std())

import pandas as pd
print("\nCorrelación entre factores (debe ser ~diagonal):")
print(pd.DataFrame(scores).corr().abs().mean().mean().round(3), "= correlación media fuera de diagonal")

## Ortogonalidad de factores

In [48]:
import numpy as np

clases = np.load("/content/clases.npy")
print("clases:", clases.shape, "| dtype:", clases.dtype)
print("Primeros 5 valores:", clases[:5])
print("Valores únicos:", np.unique(clases))

clases: (2450,) | dtype: <U22
Primeros 5 valores: ['suelo_urbano' 'vegetacion_densa' 'ozono_anomalo'
 'contaminacion_alta_SO2' 'contaminacion_alta_SO2']
Valores únicos: ['contaminacion_alta_NO2' 'contaminacion_alta_SO2' 'ozono_anomalo'
 'suelo_urbano' 'vegetacion_densa']


In [49]:
X = np.load("/content/z_img.npy")
_, idx_uniq = np.unique(X, axis=0, return_index=True)
clases_uniq = clases[idx_uniq]   # alinea las clases con las 1095 únicas
print("clases_uniq:", clases_uniq.shape)   # debe ser (1095,)

clases_uniq: (1095,)


In [50]:
print("clases tiene", clases.shape[0], "filas")  # debe ser 2450
print("idx_uniq tiene", idx_uniq.shape[0], "índices")  # debe ser 1095
print("máximo índice en idx_uniq:", idx_uniq.max())  # debe ser < 2450

clases tiene 2450 filas
idx_uniq tiene 1095 índices
máximo índice en idx_uniq: 2449


In [ ]:
scores_df = pd.DataFrame(scores, columns=[f"F{i+1}" for i in range(16)])
scores_df["clase"] = clases_uniq

perfil = scores_df.groupby("clase").mean()
print("\nPuntaje factorial medio por clase (filas=factores, columnas=clases):\n")
print(perfil.round(2).T)

In [ ]:
scores_df = pd.DataFrame(scores, columns=[f"PC{i+1}" for i in range(scores.shape[1])])
scores_df["clase"] = clases_uniq

plt.figure(figsize=(10,8))

for clase in scores_df["clase"].unique():
    subset = scores_df[scores_df["clase"] == clase]

    plt.scatter(
        subset["PC1"],
        subset["PC2"],
        alpha=0.6,
        s=20,
        label=clase
    )

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Distribución en espacio PCA")
plt.legend(bbox_to_anchor=(1.05,1), loc='upper left')
plt.grid(alpha=0.3)

plt.show()

## Separación de clases por factor

In [ ]:
from scipy import stats
import numpy as np

print("Factores ordenados por capacidad de separar clases (ANOVA F):\n")
resultados = []
for i in range(16):
    grupos = [scores[clases_uniq == c, i] for c in np.unique(clases_uniq)]
    f_stat, p_val = stats.f_oneway(*grupos)
    resultados.append((f"F{i+1}", f_stat, p_val))

for nombre, f, p in sorted(resultados, key=lambda x: -x[1]):
    marca = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
    print(f"{nombre}: F={f:6.2f}  p={p:.4f} {marca}")

## Conclusión: Independencia factor-clase

In [ ]:
import numpy as np
gases = np.load("/content/s5p_data.npy")[idx_uniq][:, [0,2,4]]
for g, nombre in enumerate(["NO2","SO2","O3"]):
    corrs = [np.corrcoef(scores[:,f], gases[:,g])[0,1] for f in range(16)]
    mejor = np.argmax(np.abs(corrs))
    print(f"{nombre}: mejor factor F{mejor+1}, r={corrs[mejor]:.3f}")

In [ ]:
for cada factor: clase top + si la diferencia es siquiera significativa
from scipy import stats
import numpy as np

perfil = scores_df.groupby("clase").mean()
for f in perfil.columns:
    grupos = [scores_df[scores_df.clase==c][f].values for c in perfil.index]
    _, p = stats.f_oneway(*grupos)
    top_clase = perfil[f].idxmax()
    sig = "SIGNIFICATIVO" if p < 0.05 else "NO significativo (ruido)"
    print(f"{f}: máx en {top_clase}, pero p={p:.3f} -> {sig}")

In [ ]:
import numpy as np

neuronas_originales = np.where(vivas)[0]

print("La columna 195 de X_afe es la neurona SAE:", neuronas_originales[195])
print("La columna 136 de X_afe es la neurona SAE:", neuronas_originales[136])

def col_de_neurona(n_original):
    pos = np.where(neuronas_originales == n_original)[0]
    if len(pos) == 0:
        return None
    return int(pos[0])

In [58]:
muertas = [5, 208, 308, 316, 431, 488, 505]
print("¿Alguna muerta sobrevivió?", any(m in neuronas_originales for m in muertas))  # debe ser False
print("Total vivas:", len(neuronas_originales))  # debe ser 505

¿Alguna muerta sobrevivió? False
Total vivas: 505


In [ ]:
from scipy import stats
n = 137
acts_por_clase = [X_afe[clases_uniq==c, n] for c in np.unique(clases_uniq)]
f_stat, p = stats.f_oneway(*acts_por_clase)
print(f"Neurona {n}: F={f_stat:.2f}, p={p:.4f}")

In [ ]:
from scipy import stats
n = 196
acts_por_clase = [X_afe[clases_uniq==c, n] for c in np.unique(clases_uniq)]
f_stat, p = stats.f_oneway(*acts_por_clase)
print(f"Neurona {n}: F={f_stat:.2f}, p={p:.4f}")

In [ ]:
!pip install semopy
import semopy

In [62]:
loadings = pd.DataFrame(fa.loadings_, columns=[f"F{i+1}" for i in range(16)])

In [63]:
rotated_loadings = pd.DataFrame(fa.loadings_, columns=[f"F{i+1}" for i in range(fa.n_factors)])
rotated_loadings["comunalidad"] = fa.get_communalities()
display(rotated_loadings)

,F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,F11,F12,F13,F14,F15,F16,comunalidad
0,0.357736,-0.130862,-0.068055,0.078814,-0.238858,-0.090925,-0.625807,-0.046278,0.023932,-0.093302,0.010086,-0.165843,-0.233734,0.029976,-0.179125,-0.087039,0.747115
1,-0.138386,-0.015643,-0.222953,0.344906,-0.073952,-0.067943,0.098490,0.143502,-0.022909,-0.133296,-0.383656,0.140603,0.284189,0.372358,-0.100884,-0.278975,0.721114
2,0.040320,0.161909,-0.022059,-0.071814,0.096012,0.633249,-0.064885,-0.017811,-0.061482,0.039468,0.026663,-0.061832,0.002493,-0.067500,0.601156,-0.091947,0.832511
3,-0.629888,0.067611,-0.014993,-0.121215,-0.072154,-0.129429,0.005211,0.110135,-0.137015,0.041281,-0.080762,-0.076921,-0.082331,-0.009102,0.581661,0.013909,0.828665
4,0.362551,0.090937,0.409063,-0.136498,-0.376919,0.148850,-0.198455,-0.118680,-0.291935,-0.085312,0.056805,-0.075703,-0.142204,-0.185883,-0.052721,-0.311003,0.799110
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
500,0.186383,0.153329,0.041106,0.161477,-0.201406,-0.059385,-0.001115,0.084375,0.543779,0.188884,0.055550,-0.151049,-0.121186,0.426579,0.187071,0.393018,0.880614
501,0.047569,-0.154925,-0.160669,0.471577,-0.089343,0.255246,0.048136,0.492293,-0.160353,0.093529,-0.053917,-0.191700,-0.113163,0.338420,-0.017744,0.229833,0.846856
502,-0.297933,0.374679,0.241990,-0.218070,-0.160572,0.037843,0.201295,-0.167940,0.069367,0.619122,-0.106029,-0.096175,-0.138636,-0.130234,0.181753,-0.143542,0.929635
503,0.163972,-0.537175,0.229034,-0.187326,0.439128,0.295081,-0.004254,-0.101567,-0.072309,-0.088966,0.062883,0.009895,-0.108915,-0.061189,0.018013,-0.092371,0.734892


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 10))
sns.heatmap(rotated_loadings.drop(columns=['comunalidad']), cmap='viridis', annot=True, fmt=".2f", linewidths=.5)
plt.title('Heatmap de la Matriz de Cargas Rotadas (Varimax)')
plt.xlabel('Factores')
plt.ylabel('Variables')
plt.show()

In [ ]:
for f in range(4):
    top = loadings[f"F{f+1}"].abs().sort_values(ascending=False).head(4)
    print(f"F{f+1} -> dimensiones {list(top.index)}")

In [ ]:
dims_c1 = [378, 346, 240, 459]
dims_c2 = [20, 283, 278, 159]
dims_c3 = [103, 302, 270, 106]
dims_c4 = [466, 93, 126, 332]

todas = dims_c1 + dims_c2 + dims_c3 + dims_c4
df = pd.DataFrame(X_afe[:, todas], columns=[f"d{i}" for i in todas])

## Modelo de ecuaciones estructurales (SEM)

In [ ]:
modelo = """
CargaAntropogenica =~ d378 + d346 + d240 + d459
EstresVegetal      =~ d20 + d283 + d278 + d159
DensidadUrbana     =~ d103 + d302 + d270 + d106
VolatilidadAtm     =~ d466 + d93 + d126 + d332

CargaAntropogenica ~~ EstresVegetal
CargaAntropogenica ~~ DensidadUrbana
CargaAntropogenica ~~ VolatilidadAtm
EstresVegetal ~~ DensidadUrbana
EstresVegetal ~~ VolatilidadAtm
DensidadUrbana ~~ VolatilidadAtm
"""
mod = semopy.Model(modelo)
mod.fit(df)

stats = semopy.calc_stats(mod)
print(stats[["RMSEA", "CFI", "chi2", "DoF"]].T)

In [ ]:
import numpy as np
import pandas as pd

S = df.corr()

Sigma = mod.calc_sigma()[0]

D = np.sqrt(np.diag(Sigma))
Sigma_corr = Sigma / np.outer(D, D)

Sigma_corr = pd.DataFrame(
    Sigma_corr,
    index=df.columns,
    columns=df.columns
)

resid = S - Sigma_corr

resid_abs = resid.abs()

np.fill_diagonal(resid_abs.values, 0)

errores = (
    resid_abs.stack()
    .sort_values(ascending=False)
    .drop_duplicates()
)

print(errores.head(20))

In [68]:
print(mod.inspect())

                  lval  op                rval  Estimate  Std. Err    z-value  \
0                 d378   ~  CargaAntropogenica  1.000000         -          -   
1                 d346   ~  CargaAntropogenica  1.039501  0.016524  62.910142   
2                 d240   ~  CargaAntropogenica  1.003636  0.018095  55.463538   
3                 d459   ~  CargaAntropogenica  1.034622  0.016739  61.809741   
4                  d20   ~       EstresVegetal  1.000000         -          -   
5                 d283   ~       EstresVegetal  1.001538  0.022147  45.223167   
6                 d278   ~       EstresVegetal  0.994576   0.02232  44.558958   
7                 d159   ~       EstresVegetal  0.888567  0.025144  35.338433   
8                 d103   ~      DensidadUrbana  1.000000         -          -   
9                 d302   ~      DensidadUrbana  1.001210  0.017442  57.402251   
10                d270   ~      DensidadUrbana  0.967936  0.018688  51.793412   
11                d106   ~  

In [ ]:
import numpy as np

S = df.corr().values

Sigma = mod.calc_sigma()[0]

D = np.sqrt(np.diag(Sigma))
Sigma_corr = Sigma / np.outer(D, D)

resid = S - Sigma_corr

p = S.shape[0]
srmr = np.sqrt(np.sum(np.triu(resid**2, k=1)) / (p*(p-1)/2))

print("SRMR =", round(srmr,4))

In [ ]:
import numpy as np
import pandas as pd

z_uniq = np.unique(np.load("/content/z_img.npy"), axis=0)

clases = np.unique(clases_uniq)
activacion_media = pd.DataFrame(
    {c: z_uniq[clases_uniq == c].mean(axis=0) for c in clases}
)

activacion_media["clase_top"] = activacion_media.idxmax(axis=1)
activacion_media["max"]  = activacion_media[clases].max(axis=1)
activacion_media["mean"] = activacion_media[clases].mean(axis=1)
activacion_media["selectividad"] = activacion_media["max"] / (activacion_media["mean"] + 1e-8)

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

clases = np.unique(clases_uniq)
umbral_on = 1e-6

frecuencia = pd.DataFrame({
    c: (z_uniq[clases_uniq == c] > umbral_on).mean(axis=0) for c in clases
})

frecuencia["clase_top"]    = frecuencia[clases].idxmax(axis=1)
frecuencia["freq_max"]     = frecuencia[clases].max(axis=1)
frecuencia["freq_mean"]    = frecuencia[clases].mean(axis=1)
frecuencia["selectividad"] = frecuencia["freq_max"] / (frecuencia["freq_mean"] + 1e-8)

activas = frecuencia[frecuencia["freq_max"] > 0.20].copy()
print(f"Neuronas que se encienden en >20% de su clase top: {len(activas)} de 512")

In [ ]:
selectivas = activas.sort_values("selectividad", ascending=False)
print("Distribución de selectividad entre las 453 activas:")
print(selectivas["selectividad"].describe())
print("\nTop 15 más selectivas:")
print(selectivas[["clase_top", "freq_max", "freq_mean", "selectividad"]].head(15))

In [ ]:
from scipy import stats
from statsmodels.stats.multitest import multipletests
import pandas as pd
import numpy as np

clases = np.unique(clases_uniq)
resultados = []
for n in activas.index:
    grupos = [z_uniq[clases_uniq == c, n] for c in clases]
    f, p = stats.f_oneway(*grupos)
    resultados.append({
        "neurona": n,
        "clase_top": activas.loc[n, "clase_top"],
        "freq_max": activas.loc[n, "freq_max"],
        "selectividad": activas.loc[n, "selectividad"],
        "F": f, "p": p
    })

res = pd.DataFrame(resultados)
res["p_corr"] = multipletests(res["p"], method="fdr_bh")[1]
res["significativa"] = res["p_corr"] < 0.05

print(f"Neuronas selectivas Y significativas (FDR): {res['significativa'].sum()} de {len(res)}")
print("\nPor clase:")
print(res[res["significativa"]]["clase_top"].value_counts())
print("\nLas más significativas:")
print(res[res["significativa"]].sort_values("p_corr")[["neurona","clase_top","selectividad","p_corr"]].head(10))

In [ ]:
no_cero = z_uniq[z_uniq > 1e-6]
print("Percentiles de activaciones >0:", np.percentile(no_cero, [1, 10, 50, 90, 99]))